In [1]:
import numpy as np
import torch
from DiffusionTS_utils import get_datamodule

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from Utils.context_fid import Context_FID
from Utils.metric_utils import display_scores
from Utils.cross_correlation import CrossCorrelLoss

In [3]:
!ls /workspaces/HIAAC-KR-Dev-Container/shared_data/daghar/standardized_view

KuHar	     RealWorld_thigh	 RealWorld_waist  WISDM
MotionSense  RealWorld_upperarm  UCI


In [4]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [ ]:
dataset_names = ["KuHar", "MotionSense", "RealWorld_thigh", "RealWorld_waist", "UCI", "WISDM"]
dataset_names = ["KuHar"]
metric_iterations = 5

metrics = {}
for dataset_name in dataset_names:
    dm = get_datamodule(dataset_name)
    dm.setup(stage="fit")
    original_data, original_labels = dm.train_dataloader().dataset[:]
    metrics[dataset_name] = {}
    for unique_label in np.unique(original_labels):
        print(f"Processing dataset: {dataset_name}, Unique Label: {unique_label}")
        class_data = original_data[original_labels == unique_label]
        # print(f"Dataset: {dataset_name}, Unique Label: {unique_label}, Data Shape: {class_data.shape}")
        fake_data = np.load(f"Executed/SyntheticData/daghar_kh/10/daghar_kh-trained_on-class_{unique_label}-minmax_scaled-12000_epochs/fake_data.npy")
        minimum_quantity = min(len(class_data), len(fake_data))
        class_data_min = class_data[:minimum_quantity]
        fake_data_min = fake_data[:minimum_quantity] 

        # FID score ------
        context_fid_scores = []
        for iteration in range(metric_iterations):
            # Context FID
            context_fid = Context_FID(class_data_min, fake_data_min)
            print(f'Iter {iteration}: ', 'context-fid =', context_fid)
            context_fid_scores.append(context_fid)
        # Display scores
        fid_mean, fid_sigma = display_scores(context_fid_scores)
        metrics[dataset_name][unique_label] = {
            'context_fid_mean': fid_mean,
            'context_fid_sigma': fid_sigma
        }

        # Cross Correlation score------
        x_real = torch.from_numpy(original_data)
        x_fake = torch.from_numpy(fake_data)

        correlational_scores = []
        size = int(x_real.shape[0] / metric_iterations)

        for i in range(metric_iterations):
            real_idx = random_choice(x_real.shape[0], size)
            fake_idx = random_choice(x_fake.shape[0], size)
            corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
            loss = corr.compute(x_fake[fake_idx, :, :])
            correlational_scores.append(loss.item())
            print(f'Iter {i}: ', 'cross-correlation =', loss.item())

        corr_mean, corr_sigma = display_scores(correlational_scores)
        metrics[dataset_name][unique_label].update({
            'cross_correlation_mean': corr_mean,
            'cross_correlation_sigma': corr_sigma
        })


Using DataLoader with shuffle=True
Processing dataset: KuHar, Unique Label: 0
Iter 0:  context-fid = 0.8115780485961985
Iter 1:  context-fid = 1.1578917804592743
Iter 2:  context-fid = 0.6280178002047949
Iter 3:  context-fid = 0.7168668162060805
Iter 4:  context-fid = 0.5672120230763904
Final Score:  0.7763132937085478 ± 0.288599765010213
Iter 0:  cross-correlation = 120.19700999135553 

Iter 1:  cross-correlation = 120.65433751624178 

Iter 2:  cross-correlation = 120.43588721178507 

Iter 3:  cross-correlation = 119.61838414967255 

Iter 4:  cross-correlation = 119.85424054118872 

Final Score:  120.15197188204873 ± 0.522520549756415
Processing dataset: KuHar, Unique Label: 1
Iter 0:  context-fid = 0.5927697365958583
Iter 1:  context-fid = 0.7588121191954759
Iter 2:  context-fid = 0.6168404806055986
Iter 3:  context-fid = 0.8560206442163034
Iter 4:  context-fid = 0.5549398354821776
Final Score:  0.6758765632190828 ± 0.15748574745489252
Iter 0:  cross-correlation = 166.65674257096356 

In [6]:
metrics

{'KuHar': {0: {'context_fid_mean': 0.7763132937085478,
   'context_fid_sigma': 0.288599765010213,
   'cross_correlation_mean': 120.15197188204873,
   'cross_correlation_sigma': 0.522520549756415},
  1: {'context_fid_mean': 0.6758765632190828,
   'context_fid_sigma': 0.15748574745489252,
   'cross_correlation_mean': 166.1374548169706,
   'cross_correlation_sigma': 0.9278898909732303},
  2: {'context_fid_mean': 7.912803956785245,
   'context_fid_sigma': 0.30152277803729854,
   'cross_correlation_mean': 43.22809535472171,
   'cross_correlation_sigma': 1.974117399311496},
  3: {'context_fid_mean': 12.52270809990172,
   'context_fid_sigma': 0.514302298865506,
   'cross_correlation_mean': 38.477933511500936,
   'cross_correlation_sigma': 3.0925810125113435},
  4: {'context_fid_mean': 26.691527118225714,
   'context_fid_sigma': 3.567732676780816,
   'cross_correlation_mean': 30.153350147874477,
   'cross_correlation_sigma': 0.881893680494068},
  5: {'context_fid_mean': 73.22493465757023,
   '

In [17]:
import pandas as pd

df = pd.DataFrame(metrics["KuHar"])
df.columns = ["KH-0", "KH-1", "KH-2", "KH-3", "KH-4", "KH-5"]
df

,KH-0,KH-1,KH-2,KH-3,KH-4,KH-5
context_fid_mean,0.776313,0.675877,7.912804,12.522708,26.691527,73.224935
context_fid_sigma,0.288600,0.157486,0.301523,0.514302,3.567733,2.984462
cross_correlation_mean,120.151972,166.137455,43.228095,38.477934,30.153350,22.458443
cross_correlation_sigma,0.522521,0.927890,1.974117,3.092581,0.881894,3.159419


In [19]:
df["Sine"] = [0.006, 0.000, 0.015, 0.004]
df["Stocks"] = [0.147, 0.025, 0.004, 0.001]
df

,KH-0,KH-1,KH-2,KH-3,KH-4,KH-5,Sine,Stocks
context_fid_mean,0.776313,0.675877,7.912804,12.522708,26.691527,73.224935,0.006,0.147
context_fid_sigma,0.288600,0.157486,0.301523,0.514302,3.567733,2.984462,0.000,0.025
cross_correlation_mean,120.151972,166.137455,43.228095,38.477934,30.153350,22.458443,0.015,0.004
cross_correlation_sigma,0.522521,0.927890,1.974117,3.092581,0.881894,3.159419,0.004,0.001
